# Mission: 

Des relevés minutieux ont été effectués par vos agents en 2015 et en 2016. Cependant, ces relevés sont coûteux à obtenir, et à partir de ceux déjà réalisés, vous voulez tenter de prédire les émissions de CO2 et la consommation totale d’énergie de bâtiments pour lesquels elles n’ont pas encore été mesurées

Evaluer l’intérêt de l’"ENERGY STAR Score" pour la prédiction d’émissions, qui est fastidieux à calculer avec l’approche utilisée actuellement par votre équipe.

Réaliser une courte analyse exploratoire.
Tester différents modèles de prédiction afin de répondre au mieux à la problématique


## Données entrantes: Base de données SEA Building Energy Benchmarking from Open Data from the City of Seattle

## Les champs sont séparés en quatre sections :


## Sommaire:

- Import de packages ou library pour avoir des outils de traitement
- Import des données
- Restriction des colonnes de données explicatives
- Préparation des modèles de prédiction
    - Features and target selection
    - Transformers & Pipeline
- Evaluation des modèles de prédiction
    - Régression linéaire
        - Entraînement
        - Evaluation du score
        - CrossValidation
    - Utilisation de GridsearchCV
        - Choix des grilles de paramètres
        - Entraînement
        - Evaluation des scores
        - Comparaison des scores

 
        
## Import de packages ou library pour avoir des outils de traitement

In [1]:
# Importation générale
import pandas as pd
pd.set_option('display.max_columns', None)
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import joblib

In [2]:
# Importation des librairies pour la prédiction

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import PowerTransformer

from sklearn.pipeline import Pipeline
from sklearn import linear_model
from sklearn.kernel_ridge import KernelRidge
from sklearn.svm import SVR
from sklearn import tree
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import AdaBoostRegressor


# Visualisation des diagrammes
from sklearn import set_config


In [3]:
# Permet d'élargir le notebook
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:70% !important; }</style>"))

In [4]:
## Import des données brutes et analyse de leur architecture
data = pd.read_csv('D:\\Utilisateurs\\Damien\\Documents\\Test_code\\test_python\\OCR_projets\\IML\\P3_\\data_clean.csv',sep="\t",low_memory=False)

Analyse structurelle des données

In [5]:
data.shape

(1661, 23)

In [6]:
data.head()

,YearBuilt,NumberofBuildings,NumberofFloors,PropertyGFAParking,PropertyGFABuilding(s),LargestPropertyUseTypeGFA,SecondLargestPropertyUseTypeGFA,ThirdLargestPropertyUseTypeGFA,ENERGYSTARScore,SiteEnergyUse(kBtu),SiteEnergyUseWN(kBtu),TotalGHGEmissions,BuildingType,PrimaryPropertyType,LargestPropertyUseType,SecondLargestPropertyUseType,ThirdLargestPropertyUseType,CouncilDistrictCode,NbFloorsByLargestGFA,NbFloorSquare,LargestGFASquare,LargestGFAByNbFloors,PropertyGFABuildingByNbFloors
0,1906,1.0,6.0,25920,72450,98370.0,0.0,0.0,45.0,6525887.0,6541579.0,47.24,NonResidential,Small- and Mid-Sized Office,Office,NaN,NaN,7,0.000061,36.0,9.676657e+09,14052.857143,10350.0000
1,1947,1.0,4.0,37854,155934,138672.0,47539.0,11166.0,59.0,16760217.0,16463978.0,116.84,NonResidential,Large Office,Office,Parking,Other,7,0.000029,16.0,1.922992e+10,27734.400000,31186.8000
2,2008,1.0,3.0,21410,55188,55188.0,21410.0,0.0,76.0,4476997.0,4617864.0,134.69,NonResidential,Small- and Mid-Sized Office,Office,Parking,NaN,2,0.000054,9.0,3.045715e+09,13797.000000,13797.0000
3,1981,1.0,4.0,0,186971,186977.0,115477.0,0.0,86.0,12662456.0,13575377.0,226.92,NonResidential,Large Office,Office,Parking,NaN,7,0.000021,16.0,3.496040e+10,37395.400000,37394.2000
4,2008,1.0,15.0,250000,184475,434475.0,250000.0,0.0,60.0,55030192.0,56652364.0,1891.47,NonResidential,Medical Office,Medical Office,Parking,NaN,3,0.000035,225.0,1.887685e+11,27154.687500,11529.6875


In [7]:
# list_data_quanti = ['YearBuilt','NumberofBuildings', 'NumberofFloors',
#        'PropertyGFAParking', 'PropertyGFABuilding(s)', 'LargestPropertyUseTypeGFA',
#        'SecondLargestPropertyUseTypeGFA',
#        'ThirdLargestPropertyUseTypeGFA','SiteEnergyUse(kBtu)','SiteEnergyUseWN(kBtu)', 'TotalGHGEmissions']

In [8]:
# list_data_col = data.columns
# list_data_quali = [icol for icol in list_data_col if icol not in list_data_quanti]
# list_data_quali

In [9]:
data.isnull().sum(axis=0)

YearBuilt                             0
NumberofBuildings                     0
NumberofFloors                        0
PropertyGFAParking                    0
PropertyGFABuilding(s)                0
LargestPropertyUseTypeGFA             0
SecondLargestPropertyUseTypeGFA       0
ThirdLargestPropertyUseTypeGFA        0
ENERGYSTARScore                     565
SiteEnergyUse(kBtu)                   0
SiteEnergyUseWN(kBtu)                 0
TotalGHGEmissions                     0
BuildingType                          0
PrimaryPropertyType                   0
LargestPropertyUseType                0
SecondLargestPropertyUseType        805
ThirdLargestPropertyUseType        1309
CouncilDistrictCode                   0
NbFloorsByLargestGFA                  0
NbFloorSquare                         0
LargestGFASquare                      0
LargestGFAByNbFloors                  0
PropertyGFABuildingByNbFloors         0
dtype: int64

#### Restriction des colonnes de données explicatives

In [10]:
list_data_exp = ['YearBuilt','NumberofBuildings', 'NumberofFloors', 'PropertyGFAParking',
       'PropertyGFABuilding(s)', 'LargestPropertyUseTypeGFA','SecondLargestPropertyUseTypeGFA','ThirdLargestPropertyUseTypeGFA',
       'BuildingType', 'PrimaryPropertyType', 'LargestPropertyUseType',
       'CouncilDistrictCode']

In [11]:
list_data_quanti_exp = ['YearBuilt','NumberofBuildings', 'NumberofFloors', 'PropertyGFAParking',
       'PropertyGFABuilding(s)', 'LargestPropertyUseTypeGFA','SecondLargestPropertyUseTypeGFA','ThirdLargestPropertyUseTypeGFA']

In [12]:
list_data_quali_exp = ['BuildingType', 'PrimaryPropertyType', 'LargestPropertyUseType',
       'CouncilDistrictCode']

# Préparation des modèles de prédiction

## Features and target selection

In [13]:
X = data.loc[:,list_data_exp]
y = data.loc[:,['SiteEnergyUseWN(kBtu)']]

In [14]:
n_quanti_exp = len(list_data_quanti_exp)

In [15]:
y

,SiteEnergyUseWN(kBtu)
0,6.541579e+06
1,1.646398e+07
2,4.617864e+06
3,1.357538e+07
4,5.665236e+07
...,...
1656,9.430032e+05
1657,1.053706e+06
1658,6.053764e+06
1659,7.828413e+05


In [16]:
from sklearn import model_selection

X_train, X_test, y_train, y_test = model_selection.train_test_split(X, y, test_size=0.2, random_state=0  ) # 20% des données dans le jeu de test # random_state = 1 pour avoir le même train-test set dans les autres jupyter notebooks

In [17]:
X_train.head()

,YearBuilt,NumberofBuildings,NumberofFloors,PropertyGFAParking,PropertyGFABuilding(s),LargestPropertyUseTypeGFA,SecondLargestPropertyUseTypeGFA,ThirdLargestPropertyUseTypeGFA,BuildingType,PrimaryPropertyType,LargestPropertyUseType,CouncilDistrictCode
1428,1929,1.0,4.0,0,49951,32039.0,24035.0,0.0,Nonresidential COS,Small- and Mid-Sized Office,Office,7
319,1960,1.0,1.0,0,119661,118733.0,0.0,0.0,NonResidential,Distribution Center,Distribution Center,2
799,1988,1.0,3.0,0,21118,21118.0,0.0,0.0,NonResidential,Residence Hall,Residence Hall/Dormitory,4
1326,1906,1.0,1.0,0,26440,26440.0,0.0,0.0,NonResidential,Worship Facility,Worship Facility,3
393,1966,1.0,1.0,0,55700,55700.0,0.0,0.0,NonResidential,Warehouse,Non-Refrigerated Warehouse,2


In [18]:
X_train.iloc[:,list(range(n_quanti_exp))].head()

,YearBuilt,NumberofBuildings,NumberofFloors,PropertyGFAParking,PropertyGFABuilding(s),LargestPropertyUseTypeGFA,SecondLargestPropertyUseTypeGFA,ThirdLargestPropertyUseTypeGFA
1428,1929,1.0,4.0,0,49951,32039.0,24035.0,0.0
319,1960,1.0,1.0,0,119661,118733.0,0.0,0.0
799,1988,1.0,3.0,0,21118,21118.0,0.0,0.0
1326,1906,1.0,1.0,0,26440,26440.0,0.0,0.0
393,1966,1.0,1.0,0,55700,55700.0,0.0,0.0


In [19]:
X_train.iloc[:,:n_quanti_exp].head()

,YearBuilt,NumberofBuildings,NumberofFloors,PropertyGFAParking,PropertyGFABuilding(s),LargestPropertyUseTypeGFA,SecondLargestPropertyUseTypeGFA,ThirdLargestPropertyUseTypeGFA
1428,1929,1.0,4.0,0,49951,32039.0,24035.0,0.0
319,1960,1.0,1.0,0,119661,118733.0,0.0,0.0
799,1988,1.0,3.0,0,21118,21118.0,0.0,0.0
1326,1906,1.0,1.0,0,26440,26440.0,0.0,0.0
393,1966,1.0,1.0,0,55700,55700.0,0.0,0.0


In [20]:
X_train.iloc[:,n_quanti_exp:].head()

,BuildingType,PrimaryPropertyType,LargestPropertyUseType,CouncilDistrictCode
1428,Nonresidential COS,Small- and Mid-Sized Office,Office,7
319,NonResidential,Distribution Center,Distribution Center,2
799,NonResidential,Residence Hall,Residence Hall/Dormitory,4
1326,NonResidential,Worship Facility,Worship Facility,3
393,NonResidential,Warehouse,Non-Refrigerated Warehouse,2


## Transformers & Pipeline  

In [21]:
# Scaling quantitatives data
trf1 = ColumnTransformer([("scale", StandardScaler(), list(range(n_quanti_exp)))], remainder ='passthrough',verbose_feature_names_out=False)

In [22]:
# Power transformers pour les variables qualitatives
trf2 = ColumnTransformer([('quantities', PowerTransformer(method="yeo-johnson"), list(range(n_quanti_exp)))], remainder ='passthrough',verbose_feature_names_out=False)

In [23]:
# One-Hot-Encoders pour les variables qualitatives
trf3 = ColumnTransformer([('categories', OneHotEncoder(dtype='int', handle_unknown = 'ignore'), list(range(n_quanti_exp,X.shape[1])))], remainder ='passthrough',verbose_feature_names_out=False)

In [24]:
pd.DataFrame(trf3.fit_transform(X_train).toarray()) #columns=trf1.get_feature_names_out()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96
0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1929.0,1.0,4.0,0.0,49951.0,32039.0,24035.0,0.0
1,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1960.0,1.0,1.0,0.0,119661.0,118733.0,0.0,0.0
2,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1988.0,1.0,3.0,0.0,21118.0,21118.0,0.0,0.0
3,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1906.0,1.0,1.0,0.0,26440.0,26440.0,0.0,0.0
4,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1966.0,1.0,1.0,0.0,55700.0,55700.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1323,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1906.0,1.0,3.0,0.0,27480.0,22218.0,8668.0,0.0
1324,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1969.0,1.0,1.0,0.0,25658.0,18854.0,6804.0,0.0
1325,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1994.0,1.0,1.0,0.0,20050.0,8108.0,7726.0,3779.0
1326,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0

In [25]:
trf3.get_feature_names_out()

array(['BuildingType_Campus', 'BuildingType_NonResidential',
       'BuildingType_Nonresidential COS',
       'BuildingType_Nonresidential WA', 'BuildingType_SPS-District K-12',
       'PrimaryPropertyType_Distribution Center',
       'PrimaryPropertyType_Hospital', 'PrimaryPropertyType_Hotel',
       'PrimaryPropertyType_K-12 School',
       'PrimaryPropertyType_Laboratory',
       'PrimaryPropertyType_Large Office',
       'PrimaryPropertyType_Low-Rise Multifamily',
       'PrimaryPropertyType_Medical Office',
       'PrimaryPropertyType_Mixed Use Property',
       'PrimaryPropertyType_Office', 'PrimaryPropertyType_Other',
       'PrimaryPropertyType_Refrigerated Warehouse',
       'PrimaryPropertyType_Residence Hall',
       'PrimaryPropertyType_Restaurant',
       'PrimaryPropertyType_Retail Store',
       'PrimaryPropertyType_Self-Storage Facility',
       'PrimaryPropertyType_Senior Care Community',
       'PrimaryPropertyType_Small- and Mid-Sized Office',
       'PrimaryProperty

In [26]:
len(trf3.get_feature_names_out())

97

In [27]:
# Log transformer for predicted variable y
trf4 = ColumnTransformer([("scale", PowerTransformer(method="yeo-johnson"), ['SiteEnergyUseWN(kBtu)'])], remainder ='passthrough')

# regressor = LinearRegression()
# regr = TransformedTargetRegressor(regressor=regressor, transformer= PowerTransformer(method="yeo-johnson"))
y_train_trf = pd.DataFrame(trf4.fit_transform(y_train)) #columns=trf1.get_feature_names_out()

In [28]:
y_train_trf.head()

,0
0,0.351016
1,-0.860131
2,-0.532774
3,-0.827924
4,-1.049447


In [29]:
y_train_log = np.log(1 + y_train + np.abs(np.min(y_train)))
y_train_log

C:\Users\daims\anaconda3\lib\site-packages\numpy\core\fromnumeric.py:85: FutureWarning: In a future version, DataFrame.min(axis=None) will return a scalar min over the entire DataFrame. To retain the old behavior, use 'frame.min(axis=0)' or just 'frame.min()'
  return reduction(axis=axis, out=out, **passkwargs)


,SiteEnergyUseWN(kBtu)
1428,15.553123
319,13.561734
799,14.171537
1326,13.624684
393,13.177073
...,...
835,14.010384
1216,13.877509
1653,14.505221
559,16.956574


In [30]:
y_train = y_train_log

In [31]:
# Make pipeline

# Pipelines linéaires
pipe_tf_lr = Pipeline(steps =[('tf1', trf1),('tf2', trf2),('tf3', trf3),('model_lr', linear_model.LinearRegression())])
pipe_tf_ridge = Pipeline(steps =[('tf1', trf1),('tf2', trf2),('tf3', trf3),('model_ridge', linear_model.Ridge())])
pipe_tf_lasso = Pipeline(steps =[('tf1', trf1),('tf2', trf2),('tf3', trf3),('model_lasso', linear_model.Lasso())])
pipe_tf_enet = Pipeline(steps =[('tf1', trf1),('tf2', trf2),('tf3', trf3),('model_enet', linear_model.ElasticNet())]) #alpha and l1_ratio

# Pipelines non linéaires
pipe_tf_krr = Pipeline(steps =[('tf1', trf1),('tf2', trf2),('tf3', trf3),('model_krr', KernelRidge(kernel="rbf"))]) #(kernel="rbf", gamma=0.1)
pipe_tf_svr = Pipeline(steps =[('tf1', trf1),('tf2', trf2),('tf3', trf3),('model_svr', SVR(kernel="rbf"))]) #(kernel="rbf", gamma=0.1)
pipe_tf_tree = Pipeline(steps =[('tf1', trf1),('tf2', trf2),('tf3', trf3),('model_tree', tree.DecisionTreeRegressor())]) #max_depth=
pipe_tf_forest = Pipeline(steps =[('tf1', trf1),('tf2', trf2),('tf3', trf3),('model_forest', RandomForestRegressor())]) #max_depth=, n_estimators
pipe_tf_xgb = Pipeline(steps =[('tf1', trf1),('tf2', trf2),('tf3', trf3),('model_xgb', GradientBoostingRegressor())]) #n_estimators=100, learning_rate=0.1, max_depth=1, random_state=0,
pipe_tf_ada = Pipeline(steps =[('tf1', trf1),('tf2', trf2),('tf3', trf3),('model_ada', AdaBoostRegressor())]) #n_estimators=100, learning_rate=0.1, max_depth=1, random_state=0,


In [32]:
# Visualisation par diagrammes
set_config(display="diagram")
#pipe_tf_lr

# Evaluation des modèles de prédiction  
## Régression linéaire  
### Entraînement

In [33]:
# Entraînement des pipelines linéaires
pipe_tf_lr.fit(X_train, y_train)

Pipeline(steps=[('tf1',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('scale', StandardScaler(),
                                                  [0, 1, 2, 3, 4, 5, 6, 7])],
                                   verbose_feature_names_out=False)),
                ('tf2',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('quantities',
                                                  PowerTransformer(),
                                                  [0, 1, 2, 3, 4, 5, 6, 7])],
                                   verbose_feature_names_out=False)),
                ('tf3',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('categories',
                                                  OneHotEncoder(dtype='int',
                                                                handle_unknown='ignore'),
                                                  [8, 9, 10, 11])],
                                   verbose_feature_names_out=False)),
                ('model_lr', LinearRegression())])

#### Evaluation du score

In [34]:
# trf1.get_feature_names_out()

In [35]:
tab_score = pd.DataFrame({'Train set score':[pipe_tf_lr.score(X_train, y_train)]
                         },
            index = ['LR'])

In [36]:
tab_score

,Train set score
LR,0.354356


Le score R^2 déterminé est très moyen. On décide d'entrainer le modèle en utilisant une validation croisée.

#### CrossValidation

In [37]:
from sklearn.model_selection import cross_val_score
n_folds = 5

In [38]:
cv_score_lr = cross_val_score(pipe_tf_lr, X_train, y_train, cv=n_folds)

In [39]:
tab_score_cv = pd.DataFrame([cv_score_lr],
            index = ['LR'])
tab_score_cv['mean'] = [np.mean(value) for value in (tab_score_cv.loc[idx] for idx in tab_score_cv.index)]

In [40]:
tab_score_cv

,0,1,2,3,4,mean
LR,0.300416,0.318204,0.150786,0.195227,0.124045,0.217736


La variance des scores entre différents folds de cross validation semble très étalée, et donc les résultats sont peu réguliés d'un folds à l'autre. Les résultas ne sont pas très intéressants, certains donnent des scores négatifs.
Regardons les coefficients calculés sur l'apprentissage du dataset d'entraînement total:

In [41]:
pipe_tf_lr['model_lr'].coef_

array([[ 2.32422233e-01,  1.19461421e-01,  4.36647704e-01,
         1.49942288e+00, -2.28795424e+00,  2.78842579e-01,
         7.47511518e-01,  1.49017579e-01,  4.64596684e-02,
         1.17960500e+00, -3.79172950e-01, -6.84350378e-01,
        -6.10625475e-01,  4.35820967e-02, -7.90190468e-01,
        -1.64115858e-01, -1.88315662e-02,  3.16427012e-02,
        -2.62047776e-01,  5.50317102e-02, -6.32297063e-01,
         3.24560667e-01,  5.23796693e-02,  2.14194177e+00,
        -6.63516660e-01, -4.07735958e-01, -4.37690805e-01,
         1.35208748e-01, -1.82618481e-01,  4.41741611e-01,
        -6.63516660e-01, -1.68863293e+01,  1.34261114e+00,
         2.86990732e+00, -1.12611643e+00,  4.95086078e-01,
         6.50182526e-01,  8.16437128e-01, -1.08989739e+00,
         7.47511518e-01,  1.49017579e-01,  4.64596684e-02,
         4.57474944e-01,  1.51849943e-02,  1.47956094e+00,
         2.99777099e-01,  9.67151422e-01,  9.60914021e-01,
        -1.72340916e-01,  6.11013592e-01, -3.76995834e-0

Les coefficients de la régression linéaires sont en valeur absolu assez élevés, et mériteraient d'être régularisés pour diminuer l'importance des variables qui pourraient agir de manière néfaste sur la justesse de la prédiction recherchée.  

Pour réaliser cette régularisation, on va évaluer d'autres modèles de prédiction sur des grilles de paramètres étendues, et sélectionner le meilleur modèle configuré avec les hyperparamètres optimaux pour notre étude.

## Utilisation de GridSearchCV  
### Choix des grilles de paramètres

In [42]:
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_squared_log_error

def r2_exp(y_true, y_pred):
    result_score = r2_score(np.exp(y_true), np.exp(y_pred))
    return result_score

def rmse_exp(y_true, y_pred):
    result_error = mean_squared_error(np.exp(y_true), np.exp(y_pred))
    return result_error

In [43]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer

# -------------- pipelines parameters

n_folds = 5
verbose_gd = 2

alpha_range = np.logspace(-3, 2, 20) #(-2, 2, 20)
l1_ratio_range = np.linspace(0,1,11)
gamma_range = np.logspace(-8, 1, 80) #-4,1,50
gamma_svr = ['scale'] 
c_range = np.logspace(0, 10, 10) 
max_depth_range = list(range(1,20)) # 50 c'est trop grand ! 30
learning_rate_range = np.linspace(0.001,1,20) # ne devrait pas être nul #np.linspace(0,1,11)
n_estimators_range = list(range(10,1200,200))


# -------------- Score and error functions
score_r2_exp = make_scorer(r2_exp, greater_is_better=True)
loss_rmse_exp = make_scorer(rmse_exp, greater_is_better=False)

# scoring_model = ['score_r2_exp','loss_rmse_exp'] #['r2','neg_mean_squared_error'] #,'neg_mean_squared_log_error']

scoring_model = {'score_r2_exp':score_r2_exp, 'loss_rmse_exp':loss_rmse_exp}

# -------------- param_grid for pipelines 

parameters_ridge = {'model_ridge__alpha': alpha_range }
parameters_lasso = {'model_lasso__alpha': alpha_range }
parameters_enet =  {'model_enet__alpha': 2*alpha_range,'model_enet__l1_ratio': l1_ratio_range} 

parameters_krr = {'model_krr__alpha': alpha_range, "model_krr__gamma": gamma_range} # valeur de 1/(2 * sigma**2)
parameters_svr = {"model_svr__C": c_range, "model_svr__gamma": gamma_svr} # 'kernel':['linear','rbf'],model_svr__epsilon:[0.01,0.1,1,10] #default = 0.1
parameters_tree = {'model_tree__max_depth': max_depth_range} 
parameters_forest =  {'model_forest__max_depth': max_depth_range } # min_samples_leaf
parameters_xgb = {'model_xgb__max_depth': max_depth_range,
                  'model_xgb__n_estimators':n_estimators_range,
                  'model_xgb__learning_rate': learning_rate_range} #,'model_xgb__random_state': 0} # pas de lrate neg + réduire encore max_depth ? ou estimators_range
parameters_ada = {'model_ada__n_estimators':n_estimators_range,
                  'model_ada__learning_rate': learning_rate_range} #,'model_xgb__random_state': 0}

# -------------- GridsearchCV declaration

pipe_tf_ridge_cv = GridSearchCV(pipe_tf_ridge, parameters_ridge,return_train_score=True, cv=n_folds,verbose = verbose_gd, scoring = scoring_model,refit='score_r2_exp')
pipe_tf_lasso_cv = GridSearchCV(pipe_tf_lasso, parameters_lasso,return_train_score=True, cv=n_folds,verbose = verbose_gd, scoring = scoring_model,refit='score_r2_exp') #
pipe_tf_enet_cv = GridSearchCV(pipe_tf_enet, parameters_enet,return_train_score=True, cv=n_folds,verbose = verbose_gd, scoring = scoring_model,refit='score_r2_exp') #
pipe_tf_krr_cv = GridSearchCV(pipe_tf_krr, parameters_krr,return_train_score=True, cv=n_folds,verbose = verbose_gd, scoring = scoring_model,refit='score_r2_exp') #
pipe_tf_svr_cv = GridSearchCV(pipe_tf_svr, parameters_svr,return_train_score=True, cv=n_folds,verbose = verbose_gd, scoring = scoring_model,refit='score_r2_exp') #
pipe_tf_tree_cv = GridSearchCV(pipe_tf_tree, parameters_tree,return_train_score=True, cv=n_folds,verbose = verbose_gd, scoring = scoring_model,refit='score_r2_exp') #
pipe_tf_forest_cv = GridSearchCV(pipe_tf_forest, parameters_forest,return_train_score=True, cv=n_folds,verbose = verbose_gd, scoring = scoring_model,refit='score_r2_exp') 
pipe_tf_xgb_cv = GridSearchCV(pipe_tf_xgb, parameters_xgb,return_train_score=True, cv=n_folds,verbose = verbose_gd, scoring = scoring_model,refit='score_r2_exp') 
pipe_tf_ada_cv = GridSearchCV(pipe_tf_ada, parameters_ada,return_train_score=True, cv=n_folds,verbose = verbose_gd, scoring = scoring_model,refit='score_r2_exp') 


### Entraînement

In [44]:
pipe_tf_ridge_cv.fit(X_train, y_train)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
[CV] END ...........................model_ridge__alpha=0.001; total time=   0.0s
[CV] END ...........................model_ridge__alpha=0.001; total time=   0.0s
[CV] END ...........................model_ridge__alpha=0.001; total time=   0.0s
[CV] END ...........................model_ridge__alpha=0.001; total time=   0.0s
[CV] END ...........................model_ridge__alpha=0.001; total time=   0.0s
[CV] END ...........model_ridge__alpha=0.0018329807108324356; total time=   0.0s
[CV] END ...........model_ridge__alpha=0.0018329807108324356; total time=   0.0s
[CV] END ...........model_ridge__alpha=0.0018329807108324356; total time=   0.0s
[CV] END ...........model_ridge__alpha=0.0018329807108324356; total time=   0.0s
[CV] END ...........model_ridge__alpha=0.0018329807108324356; total time=   0.0s
[CV] END ............model_ridge__alpha=0.003359818286283781; total time=   0.0s
[CV] END ............model_ridge__alpha=0.00335

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('tf1',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('scale',
                                                                         StandardScaler(),
                                                                         [0, 1,
                                                                          2, 3,
                                                                          4, 5,
                                                                          6,
                                                                          7])],
                                                          verbose_feature_names_out=False)),
                                       ('tf2',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('quantities',
                                                                         PowerTransformer(),
                                                                         [0, 1,
                                                                          2, 3,
                                                                          4, 5,
                                                                          6,
                                                                          7])],
                                                          verbose_feature_names_out=False)),
                                       ('tf3',
                                        Column...
       1.12883789e-02, 2.06913808e-02, 3.79269019e-02, 6.95192796e-02,
       1.27427499e-01, 2.33572147e-01, 4.28133240e-01, 7.84759970e-01,
       1.43844989e+00, 2.63665090e+00, 4.83293024e+00, 8.85866790e+00,
       1.62377674e+01, 2.97635144e+01, 5.45559478e+01, 1.00000000e+02])},
             refit='score_r2_exp', return_train_score=True,
             scoring={'loss_rmse_exp': make_scorer(rmse_exp, greater_is_better=False),
                      'score_r2_exp': make_scorer(r2_exp)},
             verbose=2)

In [45]:
pipe_tf_lasso_cv.fit(X_train, y_train)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
[CV] END ...........................model_lasso__alpha=0.001; total time=   0.0s
[CV] END ...........................model_lasso__alpha=0.001; total time=   0.0s
[CV] END ...........................model_lasso__alpha=0.001; total time=   0.0s
[CV] END ...........................model_lasso__alpha=0.001; total time=   0.0s
[CV] END ...........................model_lasso__alpha=0.001; total time=   0.0s
[CV] END ...........model_lasso__alpha=0.0018329807108324356; total time=   0.0s
[CV] END ...........model_lasso__alpha=0.0018329807108324356; total time=   0.0s
[CV] END ...........model_lasso__alpha=0.0018329807108324356; total time=   0.0s
[CV] END ...........model_lasso__alpha=0.0018329807108324356; total time=   0.0s
[CV] END ...........model_lasso__alpha=0.0018329807108324356; total time=   0.0s
[CV] END ............model_lasso__alpha=0.003359818286283781; total time=   0.0s
[CV] END ............model_lasso__alpha=0.00335

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('tf1',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('scale',
                                                                         StandardScaler(),
                                                                         [0, 1,
                                                                          2, 3,
                                                                          4, 5,
                                                                          6,
                                                                          7])],
                                                          verbose_feature_names_out=False)),
                                       ('tf2',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('quantities',
                                                                         PowerTransformer(),
                                                                         [0, 1,
                                                                          2, 3,
                                                                          4, 5,
                                                                          6,
                                                                          7])],
                                                          verbose_feature_names_out=False)),
                                       ('tf3',
                                        Column...
       1.12883789e-02, 2.06913808e-02, 3.79269019e-02, 6.95192796e-02,
       1.27427499e-01, 2.33572147e-01, 4.28133240e-01, 7.84759970e-01,
       1.43844989e+00, 2.63665090e+00, 4.83293024e+00, 8.85866790e+00,
       1.62377674e+01, 2.97635144e+01, 5.45559478e+01, 1.00000000e+02])},
             refit='score_r2_exp', return_train_score=True,
             scoring={'loss_rmse_exp': make_scorer(rmse_exp, greater_is_better=False),
                      'score_r2_exp': make_scorer(r2_exp)},
             verbose=2)

In [46]:
pipe_tf_enet_cv.fit(X_train, y_train)

Fitting 5 folds for each of 220 candidates, totalling 1100 fits


C:\Users\daims\anaconda3\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:609: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 1943.2295388237642, tolerance: 0.5572408827145374
  model = cd_fast.sparse_enet_coordinate_descent(


[CV] END ..model_enet__alpha=0.002, model_enet__l1_ratio=0.0; total time=   0.4s


C:\Users\daims\anaconda3\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:609: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 1789.46869575423, tolerance: 0.5129180899046792
  model = cd_fast.sparse_enet_coordinate_descent(


[CV] END ..model_enet__alpha=0.002, model_enet__l1_ratio=0.0; total time=   0.3s


...

[CV] END ..model_enet__alpha=200.0, model_enet__l1_ratio=0.0; total time=   0.4s


C:\Users\daims\anaconda3\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:609: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 2629.0520910041582, tolerance: 0.52729875630924
  model = cd_fast.sparse_enet_coordinate_descent(


[CV] END ..model_enet__alpha=200.0, model_enet__l1_ratio=0.0; total time=   0.3s
[CV] END ..model_enet__alpha=200.0, model_enet__l1_ratio=0.1; total time=   0.0s
[CV] END ..model_enet__alpha=200.0, model_enet__l1_ratio=0.1; total time=   0.0s
[CV] END ..model_enet__alpha=200.0, model_enet__l1_ratio=0.1; total time=   0.0s
[CV] END ..model_enet__alpha=200.0, model_enet__l1_ratio=0.1; total time=   0.0s
[CV] END ..model_enet__alpha=200.0, model_enet__l1_ratio=0.1; total time=   0.0s
[CV] END ..model_enet__alpha=200.0, model_enet__l1_ratio=0.2; total time=   0.0s
[CV] END ..model_enet__alpha=200.0, model_enet__l1_ratio=0.2; total time=   0.0s
[CV] END ..model_enet__alpha=200.0, model_enet__l1_ratio=0.2; total time=   0.0s
[CV] END ..model_enet__alpha=200.0, model_enet__l1_ratio=0.2; total time=   0.0s
[CV] END ..model_enet__alpha=200.0, model_enet__l1_ratio=0.2; total time=   0.0s
[CV] END model_enet__alpha=200.0, model_enet__l1_ratio=0.30000000000000004; total time=   0.0s
[CV] END model

C:\Users\daims\anaconda3\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:609: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 2226.679657653032, tolerance: 0.6436910155459997
  model = cd_fast.sparse_enet_coordinate_descent(


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('tf1',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('scale',
                                                                         StandardScaler(),
                                                                         [0, 1,
                                                                          2, 3,
                                                                          4, 5,
                                                                          6,
                                                                          7])],
                                                          verbose_feature_names_out=False)),
                                       ('tf2',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('quantities',
                                                                         PowerTransformer(),
                                                                         [0, 1,
                                                                          2, 3,
                                                                          4, 5,
                                                                          6,
                                                                          7])],
                                                          verbose_feature_names_out=False)),
                                       ('tf3',
                                        Column...
       2.87689978e+00, 5.27330180e+00, 9.66586048e+00, 1.77173358e+01,
       3.24755348e+01, 5.95270288e+01, 1.09111896e+02, 2.00000000e+02]),
                         'model_enet__l1_ratio': array([0. , 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1. ])},
             refit='score_r2_exp', return_train_score=True,
             scoring={'loss_rmse_exp': make_scorer(rmse_exp, greater_is_better=False),
                      'score_r2_exp': make_scorer(r2_exp)},
             verbose=2)

In [47]:
pipe_tf_krr_cv.fit(X_train, y_train)

Fitting 5 folds for each of 1600 candidates, totalling 8000 fits
[CV] END .....model_krr__alpha=0.001, model_krr__gamma=1e-08; total time=   0.1s
[CV] END .....model_krr__alpha=0.001, model_krr__gamma=1e-08; total time=   0.1s
[CV] END .....model_krr__alpha=0.001, model_krr__gamma=1e-08; total time=   0.0s
[CV] END .....model_krr__alpha=0.001, model_krr__gamma=1e-08; total time=   0.0s
[CV] END .....model_krr__alpha=0.001, model_krr__gamma=1e-08; total time=   0.0s
[CV] END model_krr__alpha=0.001, model_krr__gamma=1.299942224413245e-08; total time=   0.1s
[CV] END model_krr__alpha=0.001, model_krr__gamma=1.299942224413245e-08; total time=   0.0s
[CV] END model_krr__alpha=0.001, model_krr__gamma=1.299942224413245e-08; total time=   0.0s
[CV] END model_krr__alpha=0.001, model_krr__gamma=1.299942224413245e-08; total time=   0.1s
[CV] END model_krr__alpha=0.001, model_krr__gamma=1.299942224413245e-08; total time=   0.1s
[CV] END model_krr__alpha=0.001, model_krr__gamma=1.6898497868124587e-

[CV] END model_krr__alpha=0.001, model_krr__gamma=8.643882620598262e-07; total time=   0.0s
[CV] END model_krr__alpha=0.001, model_krr__gamma=1.1236548001387515e-06; total time=   0.0s
[CV] END model_krr__alpha=0.001, model_krr__gamma=1.1236548001387515e-06; total time=   0.0s
[CV] END model_krr__alpha=0.001, model_krr__gamma=1.1236548001387515e-06; total time=   0.0s
[CV] END model_krr__alpha=0.001, model_krr__gamma=1.1236548001387515e-06; total time=   0.0s
[CV] END model_krr__alpha=0.001, model_krr__gamma=1.1236548001387515e-06; total time=   0.0s
[CV] END model_krr__alpha=0.001, model_krr__gamma=1.4606863203649888e-06; total time=   0.0s
[CV] END model_krr__alpha=0.001, model_krr__gamma=1.4606863203649888e-06; total time=   0.1s
[CV] END model_krr__alpha=0.001, model_krr__gamma=1.4606863203649888e-06; total time=   0.0s
[CV] END model_krr__alpha=0.001, model_krr__gamma=1.4606863203649888e-06; total time=   0.0s
[CV] END model_krr__alpha=0.001, model_krr__gamma=1.4606863203649888e-0

[CV] END model_krr__alpha=0.001, model_krr__gamma=9.712740198471168e-05; total time=   0.0s
[CV] END model_krr__alpha=0.001, model_krr__gamma=9.712740198471168e-05; total time=   0.0s
[CV] END model_krr__alpha=0.001, model_krr__gamma=0.00012626001098748553; total time=   0.0s
[CV] END model_krr__alpha=0.001, model_krr__gamma=0.00012626001098748553; total time=   0.0s
[CV] END model_krr__alpha=0.001, model_krr__gamma=0.00012626001098748553; total time=   0.0s
[CV] END model_krr__alpha=0.001, model_krr__gamma=0.00012626001098748553; total time=   0.0s
[CV] END model_krr__alpha=0.001, model_krr__gamma=0.00012626001098748553; total time=   0.1s
[CV] END model_krr__alpha=0.001, model_krr__gamma=0.000164130719537513; total time=   0.1s
[CV] END model_krr__alpha=0.001, model_krr__gamma=0.000164130719537513; total time=   0.1s
[CV] END model_krr__alpha=0.001, model_krr__gamma=0.000164130719537513; total time=   0.1s
[CV] END model_krr__alpha=0.001, model_krr__gamma=0.000164130719537513; total 

[CV] END model_krr__alpha=0.001, model_krr__gamma=0.010913767146512739; total time=   0.1s
[CV] END model_krr__alpha=0.001, model_krr__gamma=0.010913767146512739; total time=   0.1s
[CV] END model_krr__alpha=0.001, model_krr__gamma=0.014187266741165962; total time=   0.0s
[CV] END model_krr__alpha=0.001, model_krr__gamma=0.014187266741165962; total time=   0.1s
[CV] END model_krr__alpha=0.001, model_krr__gamma=0.014187266741165962; total time=   0.0s
[CV] END model_krr__alpha=0.001, model_krr__gamma=0.014187266741165962; total time=   0.0s
[CV] END model_krr__alpha=0.001, model_krr__gamma=0.014187266741165962; total time=   0.1s
[CV] END model_krr__alpha=0.001, model_krr__gamma=0.01844262708585533; total time=   0.1s
[CV] END model_krr__alpha=0.001, model_krr__gamma=0.01844262708585533; total time=   0.1s
[CV] END model_krr__alpha=0.001, model_krr__gamma=0.01844262708585533; total time=   0.1s
[CV] END model_krr__alpha=0.001, model_krr__gamma=0.01844262708585533; total time=   0.1s
[CV

[CV] END model_krr__alpha=0.001, model_krr__gamma=1.5941590374559964; total time=   0.1s
[CV] END model_krr__alpha=0.001, model_krr__gamma=1.5941590374559964; total time=   0.1s
[CV] END model_krr__alpha=0.001, model_krr__gamma=1.5941590374559964; total time=   0.1s
[CV] END model_krr__alpha=0.001, model_krr__gamma=1.5941590374559964; total time=   0.1s
[CV] END model_krr__alpha=0.001, model_krr__gamma=1.5941590374559964; total time=   0.1s
[CV] END model_krr__alpha=0.001, model_krr__gamma=2.0723146452190293; total time=   0.1s
[CV] END model_krr__alpha=0.001, model_krr__gamma=2.0723146452190293; total time=   0.1s
[CV] END model_krr__alpha=0.001, model_krr__gamma=2.0723146452190293; total time=   0.1s
[CV] END model_krr__alpha=0.001, model_krr__gamma=2.0723146452190293; total time=   0.1s
[CV] END model_krr__alpha=0.001, model_krr__gamma=2.0723146452190293; total time=   0.1s
[CV] END model_krr__alpha=0.001, model_krr__gamma=2.6938893095901753; total time=   0.1s
[CV] END model_krr__a

...

[CV] END model_krr__alpha=100.0, model_krr__gamma=3.0270016537634616e-07; total time=   0.0s
[CV] END model_krr__alpha=100.0, model_krr__gamma=3.0270016537634616e-07; total time=   0.0s
[CV] END model_krr__alpha=100.0, model_krr__gamma=3.934927263095845e-07; total time=   0.0s
[CV] END model_krr__alpha=100.0, model_krr__gamma=3.934927263095845e-07; total time=   0.0s
[CV] END model_krr__alpha=100.0, model_krr__gamma=3.934927263095845e-07; total time=   0.0s
[CV] END model_krr__alpha=100.0, model_krr__gamma=3.934927263095845e-07; total time=   0.0s
[CV] END model_krr__alpha=100.0, model_krr__gamma=3.934927263095845e-07; total time=   0.0s
[CV] END model_krr__alpha=100.0, model_krr__gamma=5.115178099293146e-07; total time=   0.0s
[CV] END model_krr__alpha=100.0, model_krr__gamma=5.115178099293146e-07; total time=   0.0s
[CV] END model_krr__alpha=100.0, model_krr__gamma=5.115178099293146e-07; total time=   0.0s
[CV] END model_krr__alpha=100.0, model_krr__gamma=5.115178099293146e-07; total

[CV] END model_krr__alpha=100.0, model_krr__gamma=3.401304938279246e-05; total time=   0.0s
[CV] END model_krr__alpha=100.0, model_krr__gamma=3.401304938279246e-05; total time=   0.0s
[CV] END model_krr__alpha=100.0, model_krr__gamma=3.401304938279246e-05; total time=   0.0s
[CV] END model_krr__alpha=100.0, model_krr__gamma=4.421499907374487e-05; total time=   0.0s
[CV] END model_krr__alpha=100.0, model_krr__gamma=4.421499907374487e-05; total time=   0.0s
[CV] END model_krr__alpha=100.0, model_krr__gamma=4.421499907374487e-05; total time=   0.0s
[CV] END model_krr__alpha=100.0, model_krr__gamma=4.421499907374487e-05; total time=   0.0s
[CV] END model_krr__alpha=100.0, model_krr__gamma=4.421499907374487e-05; total time=   0.0s
[CV] END model_krr__alpha=100.0, model_krr__gamma=5.747694424835358e-05; total time=   0.0s
[CV] END model_krr__alpha=100.0, model_krr__gamma=5.747694424835358e-05; total time=   0.0s
[CV] END model_krr__alpha=100.0, model_krr__gamma=5.747694424835358e-05; total t

[CV] END model_krr__alpha=100.0, model_krr__gamma=0.0038218926206331147; total time=   0.0s
[CV] END model_krr__alpha=100.0, model_krr__gamma=0.0038218926206331147; total time=   0.0s
[CV] END model_krr__alpha=100.0, model_krr__gamma=0.0038218926206331147; total time=   0.0s
[CV] END model_krr__alpha=100.0, model_krr__gamma=0.0038218926206331147; total time=   0.0s
[CV] END model_krr__alpha=100.0, model_krr__gamma=0.004968239594734388; total time=   0.0s
[CV] END model_krr__alpha=100.0, model_krr__gamma=0.004968239594734388; total time=   0.0s
[CV] END model_krr__alpha=100.0, model_krr__gamma=0.004968239594734388; total time=   0.0s
[CV] END model_krr__alpha=100.0, model_krr__gamma=0.004968239594734388; total time=   0.0s
[CV] END model_krr__alpha=100.0, model_krr__gamma=0.004968239594734388; total time=   0.0s
[CV] END model_krr__alpha=100.0, model_krr__gamma=0.006458424430196978; total time=   0.0s
[CV] END model_krr__alpha=100.0, model_krr__gamma=0.006458424430196978; total time=   

[CV] END model_krr__alpha=100.0, model_krr__gamma=0.42944879887892723; total time=   0.0s
[CV] END model_krr__alpha=100.0, model_krr__gamma=0.42944879887892723; total time=   0.0s
[CV] END model_krr__alpha=100.0, model_krr__gamma=0.42944879887892723; total time=   0.0s
[CV] END model_krr__alpha=100.0, model_krr__gamma=0.5582586268862689; total time=   0.0s
[CV] END model_krr__alpha=100.0, model_krr__gamma=0.5582586268862689; total time=   0.0s
[CV] END model_krr__alpha=100.0, model_krr__gamma=0.5582586268862689; total time=   0.0s
[CV] END model_krr__alpha=100.0, model_krr__gamma=0.5582586268862689; total time=   0.0s
[CV] END model_krr__alpha=100.0, model_krr__gamma=0.5582586268862689; total time=   0.0s
[CV] END model_krr__alpha=100.0, model_krr__gamma=0.7257039612324216; total time=   0.0s
[CV] END model_krr__alpha=100.0, model_krr__gamma=0.7257039612324216; total time=   0.0s
[CV] END model_krr__alpha=100.0, model_krr__gamma=0.7257039612324216; total time=   0.0s
[CV] END model_krr

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('tf1',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('scale',
                                                                         StandardScaler(),
                                                                         [0, 1,
                                                                          2, 3,
                                                                          4, 5,
                                                                          6,
                                                                          7])],
                                                          verbose_feature_names_out=False)),
                                       ('tf2',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('quantities',
                                                                         PowerTransformer(),
                                                                         [0, 1,
                                                                          2, 3,
                                                                          4, 5,
                                                                          6,
                                                                          7])],
                                                          verbose_feature_names_out=False)),
                                       ('tf3',
                                        Column...
       1.95496614e-01, 2.54134304e-01, 3.30359912e-01, 4.29448799e-01,
       5.58258627e-01, 7.25703961e-01, 9.43373222e-01, 1.22633068e+00,
       1.59415904e+00, 2.07231465e+00, 2.69388931e+00, 3.50190046e+00,
       4.55226828e+00, 5.91768575e+00, 7.69264957e+00, 1.00000000e+01])},
             refit='score_r2_exp', return_train_score=True,
             scoring={'loss_rmse_exp': make_scorer(rmse_exp, greater_is_better=False),
                      'score_r2_exp': make_scorer(r2_exp)},
             verbose=2)

Pour certain modèles, on utilisera __y_train.values.flatten()__ car le dataframe n'est pas bien lu par gridsearchCV, ce qui entraîne un Warning et ralenti l'apprentissage.

In [48]:
y_train.values.flatten()

array([15.55312337, 13.56173406, 14.17153749, ..., 14.50522065,
       16.95657424, 14.62627687])

In [49]:
pipe_tf_svr_cv.fit(X_train, y_train.values.flatten())

Fitting 5 folds for each of 10 candidates, totalling 50 fits
[CV] END ...........model_svr__C=1.0, model_svr__gamma=scale; total time=   0.0s
[CV] END ...........model_svr__C=1.0, model_svr__gamma=scale; total time=   0.0s
[CV] END ...........model_svr__C=1.0, model_svr__gamma=scale; total time=   0.0s
[CV] END ...........model_svr__C=1.0, model_svr__gamma=scale; total time=   0.0s
[CV] END ...........model_svr__C=1.0, model_svr__gamma=scale; total time=   0.0s
[CV] END model_svr__C=12.91549665014884, model_svr__gamma=scale; total time=   0.1s
[CV] END model_svr__C=12.91549665014884, model_svr__gamma=scale; total time=   0.0s
[CV] END model_svr__C=12.91549665014884, model_svr__gamma=scale; total time=   0.0s
[CV] END model_svr__C=12.91549665014884, model_svr__gamma=scale; total time=   0.0s
[CV] END model_svr__C=12.91549665014884, model_svr__gamma=scale; total time=   0.0s
[CV] END model_svr__C=166.81005372000593, model_svr__gamma=scale; total time=   0.2s
[CV] END model_svr__C=166.810

C:\Users\daims\anaconda3\lib\site-packages\sklearn\metrics\_regression.py:805: RuntimeWarning: overflow encountered in square
  numerator = (weight * (y_true - y_pred) ** 2).sum(axis=0, dtype=np.float64)
C:\Users\daims\anaconda3\lib\site-packages\sklearn\metrics\_regression.py:442: RuntimeWarning: overflow encountered in square
  output_errors = np.average((y_true - y_pred) ** 2, axis=0, weights=sample_weight)


[CV] END model_svr__C=59948425.03189421, model_svr__gamma=scale; total time= 6.9min
[CV] END model_svr__C=59948425.03189421, model_svr__gamma=scale; total time= 6.8min


C:\Users\daims\anaconda3\lib\site-packages\sklearn\metrics\_regression.py:805: RuntimeWarning: overflow encountered in square
  numerator = (weight * (y_true - y_pred) ** 2).sum(axis=0, dtype=np.float64)
C:\Users\daims\anaconda3\lib\site-packages\sklearn\metrics\_regression.py:442: RuntimeWarning: overflow encountered in square
  output_errors = np.average((y_true - y_pred) ** 2, axis=0, weights=sample_weight)


[CV] END model_svr__C=59948425.03189421, model_svr__gamma=scale; total time= 6.8min
[CV] END model_svr__C=59948425.03189421, model_svr__gamma=scale; total time= 8.1min


...

C:\Users\daims\AppData\Local\Temp/ipykernel_81036/4098529699.py:6: RuntimeWarning: overflow encountered in exp
  result_score = r2_score(np.exp(y_true), np.exp(y_pred))
C:\Users\daims\anaconda3\lib\site-packages\sklearn\model_selection\_validation.py:770: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "C:\Users\daims\anaconda3\lib\site-packages\sklearn\model_selection\_validation.py", line 761, in _score
    scores = scorer(estimator, X_test, y_test)
  File "C:\Users\daims\anaconda3\lib\site-packages\sklearn\metrics\_scorer.py", line 103, in __call__
    score = scorer._score(cached_call, estimator, *args, **kwargs)
  File "C:\Users\daims\anaconda3\lib\site-packages\sklearn\metrics\_scorer.py", line 264, in _score
    return self._sign * self._score_func(y_true, y_pred, **self._kwargs)
  File "C:\Users\daims\AppData\Local\Temp/ipykernel_81036/4098529699.py", line 6, in r2_e

[CV] END .model_svr__C=10000000000.0, model_svr__gamma=scale; total time= 9.1min


C:\Users\daims\AppData\Local\Temp/ipykernel_81036/4098529699.py:6: RuntimeWarning: overflow encountered in exp
  result_score = r2_score(np.exp(y_true), np.exp(y_pred))
C:\Users\daims\anaconda3\lib\site-packages\sklearn\model_selection\_validation.py:770: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "C:\Users\daims\anaconda3\lib\site-packages\sklearn\model_selection\_validation.py", line 761, in _score
    scores = scorer(estimator, X_test, y_test)
  File "C:\Users\daims\anaconda3\lib\site-packages\sklearn\metrics\_scorer.py", line 103, in __call__
    score = scorer._score(cached_call, estimator, *args, **kwargs)
  File "C:\Users\daims\anaconda3\lib\site-packages\sklearn\metrics\_scorer.py", line 264, in _score
    return self._sign * self._score_func(y_true, y_pred, **self._kwargs)
  File "C:\Users\daims\AppData\Local\Temp/ipykernel_81036/4098529699.py", line 6, in r2_e

[CV] END .model_svr__C=10000000000.0, model_svr__gamma=scale; total time=10.3min


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('tf1',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('scale',
                                                                         StandardScaler(),
                                                                         [0, 1,
                                                                          2, 3,
                                                                          4, 5,
                                                                          6,
                                                                          7])],
                                                          verbose_feature_names_out=False)),
                                       ('tf2',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('quantities',
                                                                         PowerTransformer(),
                                                                         [0, 1,
                                                                          2, 3,
                                                                          4, 5,
                                                                          6,
                                                                          7])],
                                                          verbose_feature_names_out=False)),
                                       ('tf3',
                                        Column...
             param_grid={'model_svr__C': array([1.00000000e+00, 1.29154967e+01, 1.66810054e+02, 2.15443469e+03,
       2.78255940e+04, 3.59381366e+05, 4.64158883e+06, 5.99484250e+07,
       7.74263683e+08, 1.00000000e+10]),
                         'model_svr__gamma': ['scale']},
             refit='score_r2_exp', return_train_score=True,
             scoring={'loss_rmse_exp': make_scorer(rmse_exp, greater_is_better=False),
                      'score_r2_exp': make_scorer(r2_exp)},
             verbose=2)

In [50]:
pipe_tf_tree_cv.fit(X_train, y_train)

Fitting 5 folds for each of 19 candidates, totalling 95 fits
[CV] END ............................model_tree__max_depth=1; total time=   0.0s
[CV] END ............................model_tree__max_depth=1; total time=   0.0s
[CV] END ............................model_tree__max_depth=1; total time=   0.0s
[CV] END ............................model_tree__max_depth=1; total time=   0.0s
[CV] END ............................model_tree__max_depth=1; total time=   0.0s
[CV] END ............................model_tree__max_depth=2; total time=   0.0s
[CV] END ............................model_tree__max_depth=2; total time=   0.0s
[CV] END ............................model_tree__max_depth=2; total time=   0.0s
[CV] END ............................model_tree__max_depth=2; total time=   0.0s
[CV] END ............................model_tree__max_depth=2; total time=   0.0s
[CV] END ............................model_tree__max_depth=3; total time=   0.0s
[CV] END ............................model_tree_

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('tf1',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('scale',
                                                                         StandardScaler(),
                                                                         [0, 1,
                                                                          2, 3,
                                                                          4, 5,
                                                                          6,
                                                                          7])],
                                                          verbose_feature_names_out=False)),
                                       ('tf2',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('quantities',
                                                                         PowerTransformer(),
                                                                         [0, 1,
                                                                          2, 3,
                                                                          4, 5,
                                                                          6,
                                                                          7])],
                                                          verbose_feature_names_out=False)),
                                       ('tf3',
                                        Column...
                                                                                       handle_unknown='ignore'),
                                                                         [8, 9,
                                                                          10,
                                                                          11])],
                                                          verbose_feature_names_out=False)),
                                       ('model_tree',
                                        DecisionTreeRegressor())]),
             param_grid={'model_tree__max_depth': [1, 2, 3, 4, 5, 6, 7, 8, 9,
                                                   10, 11, 12, 13, 14, 15, 16,
                                                   17, 18, 19]},
             refit='score_r2_exp', return_train_score=True,
             scoring={'loss_rmse_exp': make_scorer(rmse_exp, greater_is_better=False),
                      'score_r2_exp': make_scorer(r2_exp)},
             verbose=2)

In [51]:
pipe_tf_forest_cv.fit(X_train, y_train.values.flatten())

Fitting 5 folds for each of 19 candidates, totalling 95 fits
[CV] END ..........................model_forest__max_depth=1; total time=   0.0s
[CV] END ..........................model_forest__max_depth=1; total time=   0.0s
[CV] END ..........................model_forest__max_depth=1; total time=   0.0s
[CV] END ..........................model_forest__max_depth=1; total time=   0.0s
[CV] END ..........................model_forest__max_depth=1; total time=   0.0s
[CV] END ..........................model_forest__max_depth=2; total time=   0.1s
[CV] END ..........................model_forest__max_depth=2; total time=   0.1s
[CV] END ..........................model_forest__max_depth=2; total time=   0.1s
[CV] END ..........................model_forest__max_depth=2; total time=   0.1s
[CV] END ..........................model_forest__max_depth=2; total time=   0.1s
[CV] END ..........................model_forest__max_depth=3; total time=   0.1s
[CV] END ..........................model_forest_

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('tf1',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('scale',
                                                                         StandardScaler(),
                                                                         [0, 1,
                                                                          2, 3,
                                                                          4, 5,
                                                                          6,
                                                                          7])],
                                                          verbose_feature_names_out=False)),
                                       ('tf2',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('quantities',
                                                                         PowerTransformer(),
                                                                         [0, 1,
                                                                          2, 3,
                                                                          4, 5,
                                                                          6,
                                                                          7])],
                                                          verbose_feature_names_out=False)),
                                       ('tf3',
                                        Column...
                                                                                       handle_unknown='ignore'),
                                                                         [8, 9,
                                                                          10,
                                                                          11])],
                                                          verbose_feature_names_out=False)),
                                       ('model_forest',
                                        RandomForestRegressor())]),
             param_grid={'model_forest__max_depth': [1, 2, 3, 4, 5, 6, 7, 8, 9,
                                                     10, 11, 12, 13, 14, 15, 16,
                                                     17, 18, 19]},
             refit='score_r2_exp', return_train_score=True,
             scoring={'loss_rmse_exp': make_scorer(rmse_exp, greater_is_better=False),
                      'score_r2_exp': make_scorer(r2_exp)},
             verbose=2)

In [52]:
pipe_tf_ada_cv.fit(X_train, y_train.values.flatten())

Fitting 5 folds for each of 120 candidates, totalling 600 fits
[CV] END model_ada__learning_rate=0.001, model_ada__n_estimators=10; total time=   0.0s
[CV] END model_ada__learning_rate=0.001, model_ada__n_estimators=10; total time=   0.0s
[CV] END model_ada__learning_rate=0.001, model_ada__n_estimators=10; total time=   0.0s
[CV] END model_ada__learning_rate=0.001, model_ada__n_estimators=10; total time=   0.0s
[CV] END model_ada__learning_rate=0.001, model_ada__n_estimators=10; total time=   0.0s
[CV] END model_ada__learning_rate=0.001, model_ada__n_estimators=210; total time=   0.5s
[CV] END model_ada__learning_rate=0.001, model_ada__n_estimators=210; total time=   0.4s
[CV] END model_ada__learning_rate=0.001, model_ada__n_estimators=210; total time=   0.5s
[CV] END model_ada__learning_rate=0.001, model_ada__n_estimators=210; total time=   0.5s
[CV] END model_ada__learning_rate=0.001, model_ada__n_estimators=210; total time=   0.5s
[CV] END model_ada__learning_rate=0.001, model_ada__

[CV] END model_ada__learning_rate=0.1061578947368421, model_ada__n_estimators=810; total time=   0.1s
[CV] END model_ada__learning_rate=0.1061578947368421, model_ada__n_estimators=1010; total time=   0.1s
[CV] END model_ada__learning_rate=0.1061578947368421, model_ada__n_estimators=1010; total time=   0.1s
[CV] END model_ada__learning_rate=0.1061578947368421, model_ada__n_estimators=1010; total time=   0.1s
[CV] END model_ada__learning_rate=0.1061578947368421, model_ada__n_estimators=1010; total time=   0.1s
[CV] END model_ada__learning_rate=0.1061578947368421, model_ada__n_estimators=1010; total time=   0.1s
[CV] END model_ada__learning_rate=0.15873684210526315, model_ada__n_estimators=10; total time=   0.0s
[CV] END model_ada__learning_rate=0.15873684210526315, model_ada__n_estimators=10; total time=   0.0s
[CV] END model_ada__learning_rate=0.15873684210526315, model_ada__n_estimators=10; total time=   0.0s
[CV] END model_ada__learning_rate=0.15873684210526315, model_ada__n_estimator

[CV] END model_ada__learning_rate=0.26389473684210524, model_ada__n_estimators=410; total time=   0.0s
[CV] END model_ada__learning_rate=0.26389473684210524, model_ada__n_estimators=610; total time=   0.0s
[CV] END model_ada__learning_rate=0.26389473684210524, model_ada__n_estimators=610; total time=   0.1s
[CV] END model_ada__learning_rate=0.26389473684210524, model_ada__n_estimators=610; total time=   0.0s
[CV] END model_ada__learning_rate=0.26389473684210524, model_ada__n_estimators=610; total time=   0.0s
[CV] END model_ada__learning_rate=0.26389473684210524, model_ada__n_estimators=610; total time=   0.0s
[CV] END model_ada__learning_rate=0.26389473684210524, model_ada__n_estimators=810; total time=   0.0s
[CV] END model_ada__learning_rate=0.26389473684210524, model_ada__n_estimators=810; total time=   0.0s
[CV] END model_ada__learning_rate=0.26389473684210524, model_ada__n_estimators=810; total time=   0.0s
[CV] END model_ada__learning_rate=0.26389473684210524, model_ada__n_estim

[CV] END model_ada__learning_rate=0.4216315789473684, model_ada__n_estimators=10; total time=   0.0s
[CV] END model_ada__learning_rate=0.4216315789473684, model_ada__n_estimators=210; total time=   0.0s
[CV] END model_ada__learning_rate=0.4216315789473684, model_ada__n_estimators=210; total time=   0.0s
[CV] END model_ada__learning_rate=0.4216315789473684, model_ada__n_estimators=210; total time=   0.0s
[CV] END model_ada__learning_rate=0.4216315789473684, model_ada__n_estimators=210; total time=   0.0s
[CV] END model_ada__learning_rate=0.4216315789473684, model_ada__n_estimators=210; total time=   0.0s
[CV] END model_ada__learning_rate=0.4216315789473684, model_ada__n_estimators=410; total time=   0.0s
[CV] END model_ada__learning_rate=0.4216315789473684, model_ada__n_estimators=410; total time=   0.0s
[CV] END model_ada__learning_rate=0.4216315789473684, model_ada__n_estimators=410; total time=   0.0s
[CV] END model_ada__learning_rate=0.4216315789473684, model_ada__n_estimators=410; 

[CV] END model_ada__learning_rate=0.5267894736842105, model_ada__n_estimators=1010; total time=   0.0s
[CV] END model_ada__learning_rate=0.5267894736842105, model_ada__n_estimators=1010; total time=   0.0s
[CV] END model_ada__learning_rate=0.5267894736842105, model_ada__n_estimators=1010; total time=   0.0s
[CV] END model_ada__learning_rate=0.5267894736842105, model_ada__n_estimators=1010; total time=   0.0s
[CV] END model_ada__learning_rate=0.5793684210526315, model_ada__n_estimators=10; total time=   0.0s
[CV] END model_ada__learning_rate=0.5793684210526315, model_ada__n_estimators=10; total time=   0.0s
[CV] END model_ada__learning_rate=0.5793684210526315, model_ada__n_estimators=10; total time=   0.0s
[CV] END model_ada__learning_rate=0.5793684210526315, model_ada__n_estimators=10; total time=   0.0s
[CV] END model_ada__learning_rate=0.5793684210526315, model_ada__n_estimators=10; total time=   0.0s
[CV] END model_ada__learning_rate=0.5793684210526315, model_ada__n_estimators=210; 

[CV] END model_ada__learning_rate=0.6845263157894736, model_ada__n_estimators=610; total time=   0.0s
[CV] END model_ada__learning_rate=0.6845263157894736, model_ada__n_estimators=610; total time=   0.0s
[CV] END model_ada__learning_rate=0.6845263157894736, model_ada__n_estimators=610; total time=   0.0s
[CV] END model_ada__learning_rate=0.6845263157894736, model_ada__n_estimators=810; total time=   0.0s
[CV] END model_ada__learning_rate=0.6845263157894736, model_ada__n_estimators=810; total time=   0.0s
[CV] END model_ada__learning_rate=0.6845263157894736, model_ada__n_estimators=810; total time=   0.0s
[CV] END model_ada__learning_rate=0.6845263157894736, model_ada__n_estimators=810; total time=   0.0s
[CV] END model_ada__learning_rate=0.6845263157894736, model_ada__n_estimators=810; total time=   0.0s
[CV] END model_ada__learning_rate=0.6845263157894736, model_ada__n_estimators=1010; total time=   0.0s
[CV] END model_ada__learning_rate=0.6845263157894736, model_ada__n_estimators=101

[CV] END model_ada__learning_rate=0.8422631578947368, model_ada__n_estimators=410; total time=   0.0s
[CV] END model_ada__learning_rate=0.8422631578947368, model_ada__n_estimators=410; total time=   0.0s
[CV] END model_ada__learning_rate=0.8422631578947368, model_ada__n_estimators=410; total time=   0.0s
[CV] END model_ada__learning_rate=0.8422631578947368, model_ada__n_estimators=410; total time=   0.0s
[CV] END model_ada__learning_rate=0.8422631578947368, model_ada__n_estimators=610; total time=   0.0s
[CV] END model_ada__learning_rate=0.8422631578947368, model_ada__n_estimators=610; total time=   0.0s
[CV] END model_ada__learning_rate=0.8422631578947368, model_ada__n_estimators=610; total time=   0.0s
[CV] END model_ada__learning_rate=0.8422631578947368, model_ada__n_estimators=610; total time=   0.0s
[CV] END model_ada__learning_rate=0.8422631578947368, model_ada__n_estimators=610; total time=   0.0s
[CV] END model_ada__learning_rate=0.8422631578947368, model_ada__n_estimators=810;

[CV] END model_ada__learning_rate=1.0, model_ada__n_estimators=10; total time=   0.0s
[CV] END model_ada__learning_rate=1.0, model_ada__n_estimators=210; total time=   0.0s
[CV] END model_ada__learning_rate=1.0, model_ada__n_estimators=210; total time=   0.0s
[CV] END model_ada__learning_rate=1.0, model_ada__n_estimators=210; total time=   0.0s
[CV] END model_ada__learning_rate=1.0, model_ada__n_estimators=210; total time=   0.0s
[CV] END model_ada__learning_rate=1.0, model_ada__n_estimators=210; total time=   0.0s
[CV] END model_ada__learning_rate=1.0, model_ada__n_estimators=410; total time=   0.0s
[CV] END model_ada__learning_rate=1.0, model_ada__n_estimators=410; total time=   0.0s
[CV] END model_ada__learning_rate=1.0, model_ada__n_estimators=410; total time=   0.0s
[CV] END model_ada__learning_rate=1.0, model_ada__n_estimators=410; total time=   0.0s
[CV] END model_ada__learning_rate=1.0, model_ada__n_estimators=410; total time=   0.0s
[CV] END model_ada__learning_rate=1.0, model

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('tf1',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('scale',
                                                                         StandardScaler(),
                                                                         [0, 1,
                                                                          2, 3,
                                                                          4, 5,
                                                                          6,
                                                                          7])],
                                                          verbose_feature_names_out=False)),
                                       ('tf2',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('quantities',
                                                                         PowerTransformer(),
                                                                         [0, 1,
                                                                          2, 3,
                                                                          4, 5,
                                                                          6,
                                                                          7])],
                                                          verbose_feature_names_out=False)),
                                       ('tf3',
                                        Column...
       0.26389474, 0.31647368, 0.36905263, 0.42163158, 0.47421053,
       0.52678947, 0.57936842, 0.63194737, 0.68452632, 0.73710526,
       0.78968421, 0.84226316, 0.89484211, 0.94742105, 1.        ]),
                         'model_ada__n_estimators': [10, 210, 410, 610, 810,
                                                     1010]},
             refit='score_r2_exp', return_train_score=True,
             scoring={'loss_rmse_exp': make_scorer(rmse_exp, greater_is_better=False),
                      'score_r2_exp': make_scorer(r2_exp)},
             verbose=2)

In [ ]:
pipe_tf_xgb_cv.fit(X_train, y_train.values.flatten())
# 1h / 0.1 learning rate pour max_depth_range = list(range(1,30)), learning_rate_range = np.linspace(0,1,11) , n_estimators_range = list(range(10,1200,200))


Fitting 5 folds for each of 2280 candidates, totalling 11400 fits
[CV] END model_xgb__learning_rate=0.001, model_xgb__max_depth=1, model_xgb__n_estimators=10; total time=   0.0s
[CV] END model_xgb__learning_rate=0.001, model_xgb__max_depth=1, model_xgb__n_estimators=10; total time=   0.0s
[CV] END model_xgb__learning_rate=0.001, model_xgb__max_depth=1, model_xgb__n_estimators=10; total time=   0.0s
[CV] END model_xgb__learning_rate=0.001, model_xgb__max_depth=1, model_xgb__n_estimators=10; total time=   0.0s
[CV] END model_xgb__learning_rate=0.001, model_xgb__max_depth=1, model_xgb__n_estimators=10; total time=   0.0s
[CV] END model_xgb__learning_rate=0.001, model_xgb__max_depth=1, model_xgb__n_estimators=210; total time=   0.1s
[CV] END model_xgb__learning_rate=0.001, model_xgb__max_depth=1, model_xgb__n_estimators=210; total time=   0.0s
[CV] END model_xgb__learning_rate=0.001, model_xgb__max_depth=1, model_xgb__n_estimators=210; total time=   0.1s
[CV] END model_xgb__learning_rate=0

[CV] END model_xgb__learning_rate=0.001, model_xgb__max_depth=3, model_xgb__n_estimators=410; total time=   0.5s
[CV] END model_xgb__learning_rate=0.001, model_xgb__max_depth=3, model_xgb__n_estimators=410; total time=   0.5s
[CV] END model_xgb__learning_rate=0.001, model_xgb__max_depth=3, model_xgb__n_estimators=410; total time=   0.5s
[CV] END model_xgb__learning_rate=0.001, model_xgb__max_depth=3, model_xgb__n_estimators=610; total time=   0.8s
[CV] END model_xgb__learning_rate=0.001, model_xgb__max_depth=3, model_xgb__n_estimators=610; total time=   0.8s
[CV] END model_xgb__learning_rate=0.001, model_xgb__max_depth=3, model_xgb__n_estimators=610; total time=   0.8s
[CV] END model_xgb__learning_rate=0.001, model_xgb__max_depth=3, model_xgb__n_estimators=610; total time=   0.8s
[CV] END model_xgb__learning_rate=0.001, model_xgb__max_depth=3, model_xgb__n_estimators=610; total time=   0.8s
[CV] END model_xgb__learning_rate=0.001, model_xgb__max_depth=3, model_xgb__n_estimators=810; to

[CV] END model_xgb__learning_rate=0.001, model_xgb__max_depth=5, model_xgb__n_estimators=1010; total time=   2.4s
[CV] END model_xgb__learning_rate=0.001, model_xgb__max_depth=5, model_xgb__n_estimators=1010; total time=   2.4s
[CV] END model_xgb__learning_rate=0.001, model_xgb__max_depth=5, model_xgb__n_estimators=1010; total time=   2.6s
[CV] END model_xgb__learning_rate=0.001, model_xgb__max_depth=5, model_xgb__n_estimators=1010; total time=   2.5s
[CV] END model_xgb__learning_rate=0.001, model_xgb__max_depth=5, model_xgb__n_estimators=1010; total time=   2.6s
[CV] END model_xgb__learning_rate=0.001, model_xgb__max_depth=6, model_xgb__n_estimators=10; total time=   0.0s
[CV] END model_xgb__learning_rate=0.001, model_xgb__max_depth=6, model_xgb__n_estimators=10; total time=   0.0s
[CV] END model_xgb__learning_rate=0.001, model_xgb__max_depth=6, model_xgb__n_estimators=10; total time=   0.0s
[CV] END model_xgb__learning_rate=0.001, model_xgb__max_depth=6, model_xgb__n_estimators=10; t

[CV] END model_xgb__learning_rate=0.001, model_xgb__max_depth=8, model_xgb__n_estimators=210; total time=   1.0s
[CV] END model_xgb__learning_rate=0.001, model_xgb__max_depth=8, model_xgb__n_estimators=210; total time=   1.0s
[CV] END model_xgb__learning_rate=0.001, model_xgb__max_depth=8, model_xgb__n_estimators=410; total time=   1.8s
[CV] END model_xgb__learning_rate=0.001, model_xgb__max_depth=8, model_xgb__n_estimators=410; total time=   1.9s
[CV] END model_xgb__learning_rate=0.001, model_xgb__max_depth=8, model_xgb__n_estimators=410; total time=   2.0s
[CV] END model_xgb__learning_rate=0.001, model_xgb__max_depth=8, model_xgb__n_estimators=410; total time=   2.0s
[CV] END model_xgb__learning_rate=0.001, model_xgb__max_depth=8, model_xgb__n_estimators=410; total time=   2.0s
[CV] END model_xgb__learning_rate=0.001, model_xgb__max_depth=8, model_xgb__n_estimators=610; total time=   2.8s
[CV] END model_xgb__learning_rate=0.001, model_xgb__max_depth=8, model_xgb__n_estimators=610; to

[CV] END model_xgb__learning_rate=0.001, model_xgb__max_depth=10, model_xgb__n_estimators=810; total time=   5.1s
[CV] END model_xgb__learning_rate=0.001, model_xgb__max_depth=10, model_xgb__n_estimators=810; total time=   5.2s
[CV] END model_xgb__learning_rate=0.001, model_xgb__max_depth=10, model_xgb__n_estimators=810; total time=   5.5s
[CV] END model_xgb__learning_rate=0.001, model_xgb__max_depth=10, model_xgb__n_estimators=810; total time=   5.8s
[CV] END model_xgb__learning_rate=0.001, model_xgb__max_depth=10, model_xgb__n_estimators=1010; total time=   6.6s
[CV] END model_xgb__learning_rate=0.001, model_xgb__max_depth=10, model_xgb__n_estimators=1010; total time=   6.5s
[CV] END model_xgb__learning_rate=0.001, model_xgb__max_depth=10, model_xgb__n_estimators=1010; total time=   6.7s
[CV] END model_xgb__learning_rate=0.001, model_xgb__max_depth=10, model_xgb__n_estimators=1010; total time=   7.0s
[CV] END model_xgb__learning_rate=0.001, model_xgb__max_depth=10, model_xgb__n_estim

...

[CV] END model_xgb__learning_rate=0.3164736842105263, model_xgb__max_depth=8, model_xgb__n_estimators=810; total time=   3.1s
[CV] END model_xgb__learning_rate=0.3164736842105263, model_xgb__max_depth=8, model_xgb__n_estimators=810; total time=   2.5s
[CV] END model_xgb__learning_rate=0.3164736842105263, model_xgb__max_depth=8, model_xgb__n_estimators=810; total time=   2.9s
[CV] END model_xgb__learning_rate=0.3164736842105263, model_xgb__max_depth=8, model_xgb__n_estimators=1010; total time=   2.5s
[CV] END model_xgb__learning_rate=0.3164736842105263, model_xgb__max_depth=8, model_xgb__n_estimators=1010; total time=   3.3s
[CV] END model_xgb__learning_rate=0.3164736842105263, model_xgb__max_depth=8, model_xgb__n_estimators=1010; total time=   3.3s
[CV] END model_xgb__learning_rate=0.3164736842105263, model_xgb__max_depth=8, model_xgb__n_estimators=1010; total time=   2.6s
[CV] END model_xgb__learning_rate=0.3164736842105263, model_xgb__max_depth=8, model_xgb__n_estimators=1010; total 

[CV] END model_xgb__learning_rate=0.3164736842105263, model_xgb__max_depth=10, model_xgb__n_estimators=1010; total time=   3.8s
[CV] END model_xgb__learning_rate=0.3164736842105263, model_xgb__max_depth=10, model_xgb__n_estimators=1010; total time=   2.8s
[CV] END model_xgb__learning_rate=0.3164736842105263, model_xgb__max_depth=10, model_xgb__n_estimators=1010; total time=   4.4s
[CV] END model_xgb__learning_rate=0.3164736842105263, model_xgb__max_depth=11, model_xgb__n_estimators=10; total time=   0.0s
[CV] END model_xgb__learning_rate=0.3164736842105263, model_xgb__max_depth=11, model_xgb__n_estimators=10; total time=   0.0s
[CV] END model_xgb__learning_rate=0.3164736842105263, model_xgb__max_depth=11, model_xgb__n_estimators=10; total time=   0.0s
[CV] END model_xgb__learning_rate=0.3164736842105263, model_xgb__max_depth=11, model_xgb__n_estimators=10; total time=   0.0s
[CV] END model_xgb__learning_rate=0.3164736842105263, model_xgb__max_depth=11, model_xgb__n_estimators=10; total

[CV] END model_xgb__learning_rate=0.3164736842105263, model_xgb__max_depth=13, model_xgb__n_estimators=10; total time=   0.0s
[CV] END model_xgb__learning_rate=0.3164736842105263, model_xgb__max_depth=13, model_xgb__n_estimators=10; total time=   0.0s
[CV] END model_xgb__learning_rate=0.3164736842105263, model_xgb__max_depth=13, model_xgb__n_estimators=210; total time=   1.5s
[CV] END model_xgb__learning_rate=0.3164736842105263, model_xgb__max_depth=13, model_xgb__n_estimators=210; total time=   1.4s
[CV] END model_xgb__learning_rate=0.3164736842105263, model_xgb__max_depth=13, model_xgb__n_estimators=210; total time=   1.4s
[CV] END model_xgb__learning_rate=0.3164736842105263, model_xgb__max_depth=13, model_xgb__n_estimators=210; total time=   1.5s
[CV] END model_xgb__learning_rate=0.3164736842105263, model_xgb__max_depth=13, model_xgb__n_estimators=210; total time=   1.5s
[CV] END model_xgb__learning_rate=0.3164736842105263, model_xgb__max_depth=13, model_xgb__n_estimators=410; total

[CV] END model_xgb__learning_rate=0.3164736842105263, model_xgb__max_depth=15, model_xgb__n_estimators=210; total time=   1.5s
[CV] END model_xgb__learning_rate=0.3164736842105263, model_xgb__max_depth=15, model_xgb__n_estimators=210; total time=   1.5s
[CV] END model_xgb__learning_rate=0.3164736842105263, model_xgb__max_depth=15, model_xgb__n_estimators=410; total time=   1.4s
[CV] END model_xgb__learning_rate=0.3164736842105263, model_xgb__max_depth=15, model_xgb__n_estimators=410; total time=   2.3s
[CV] END model_xgb__learning_rate=0.3164736842105263, model_xgb__max_depth=15, model_xgb__n_estimators=410; total time=   2.3s
[CV] END model_xgb__learning_rate=0.3164736842105263, model_xgb__max_depth=15, model_xgb__n_estimators=410; total time=   1.5s
[CV] END model_xgb__learning_rate=0.3164736842105263, model_xgb__max_depth=15, model_xgb__n_estimators=410; total time=   2.3s
[CV] END model_xgb__learning_rate=0.3164736842105263, model_xgb__max_depth=15, model_xgb__n_estimators=610; tot

[CV] END model_xgb__learning_rate=0.3164736842105263, model_xgb__max_depth=17, model_xgb__n_estimators=410; total time=   1.3s
[CV] END model_xgb__learning_rate=0.3164736842105263, model_xgb__max_depth=17, model_xgb__n_estimators=410; total time=   2.4s
[CV] END model_xgb__learning_rate=0.3164736842105263, model_xgb__max_depth=17, model_xgb__n_estimators=610; total time=   1.4s
[CV] END model_xgb__learning_rate=0.3164736842105263, model_xgb__max_depth=17, model_xgb__n_estimators=610; total time=   3.2s
[CV] END model_xgb__learning_rate=0.3164736842105263, model_xgb__max_depth=17, model_xgb__n_estimators=610; total time=   3.2s
[CV] END model_xgb__learning_rate=0.3164736842105263, model_xgb__max_depth=17, model_xgb__n_estimators=610; total time=   1.4s
[CV] END model_xgb__learning_rate=0.3164736842105263, model_xgb__max_depth=17, model_xgb__n_estimators=610; total time=   3.4s
[CV] END model_xgb__learning_rate=0.3164736842105263, model_xgb__max_depth=17, model_xgb__n_estimators=810; tot

In [ ]:
pd.DataFrame(pipe_tf_svr_cv.cv_results_).iloc[450:500]

In [ ]:
pd.DataFrame(pipe_tf_ridge_cv.cv_results_)

In [ ]:
pd.DataFrame(pipe_tf_svr_cv.cv_results_)

### Evaluation des scores

In [ ]:
def plot_results(estimator,estimator_name):
    plt.figure(figsize=(18,14))
    #plt.suptitle(str(f'{estimator=}'.split('=')[0]))
    plt.suptitle(estimator_name, fontsize=20)
    ax = plt.gca()

    plt.subplot(3,1,1)
    plt.plot(list(range(len(estimator.cv_results_['mean_train_score'].data))), estimator.cv_results_['mean_train_score'])
    plt.plot(list(range(len(estimator.cv_results_['mean_test_score'].data))), estimator.cv_results_['mean_test_score'])
    plt.legend(['mean_train_score','mean_test_score'])
    plt.grid()
    plt.xlabel('Iteration')
    plt.ylabel('Scores')
    
    plt.subplot(3,1,2)
    plt.plot(list(range(len(estimator.cv_results_['std_train_score'].data))), estimator.cv_results_['std_train_score'])
    plt.plot(list(range(len(estimator.cv_results_['std_test_score'].data))), estimator.cv_results_['std_test_score'])
    plt.legend(['std_train_score','std_test_score'])
    plt.grid()
    plt.xlabel('Iteration')
    plt.ylabel('Scores')

    plt.subplot(3,1,3)
    plt.plot(list(range(len(estimator.cv_results_['mean_train_score'].data))), estimator.cv_results_['mean_fit_time'])
    plt.legend(['mean_fit_time'])
    plt.grid()
    plt.xlabel('Iteration')
    plt.ylabel('Scores')
    
    # plt.axis('tight')
    plt.show()


In [ ]:
def plot_results2(estimator,estimator_name,score_name):
    plt.figure(figsize=(18,14))
    #plt.suptitle(str(f'{estimator=}'.split('=')[0]))
    plt.suptitle(estimator_name, fontsize=20)
    ax = plt.gca()
    
    mean_train = 'mean_train_' + score_name
    mean_test = 'mean_test_' + score_name
    std_train = 'std_train_' + score_name
    std_test =  'std_test_' + score_name
    
    plt.subplot(3,1,1)
    plt.plot(list(range(len(estimator.cv_results_[mean_train].data))), estimator.cv_results_[mean_train])
    plt.plot(list(range(len(estimator.cv_results_[mean_test].data))), estimator.cv_results_[mean_test])
    plt.legend([mean_train,mean_test])
    plt.grid()
    plt.xlabel('Iteration')
    plt.ylabel('Scores')
    
    plt.subplot(3,1,2)
    plt.plot(list(range(len(estimator.cv_results_[std_train].data))), estimator.cv_results_[std_train])
    plt.plot(list(range(len(estimator.cv_results_[std_test].data))), estimator.cv_results_[std_test])
    plt.legend([std_train,std_test])
    plt.grid()
    plt.xlabel('Iteration')
    plt.ylabel('Scores')

    plt.subplot(3,1,3)
    plt.plot(list(range(len(estimator.cv_results_['mean_fit_time'].data))), estimator.cv_results_['mean_fit_time'])
    plt.legend(['mean_fit_time'])
    plt.grid()
    plt.xlabel('Iteration')
    plt.ylabel('Scores')
    
    # plt.axis('tight')
    plt.show()


In [ ]:
plot_results2(pipe_tf_ridge_cv,'RIDGE','score_r2_exp')

In [ ]:
plot_results2(pipe_tf_lasso_cv,'LASSO','score_r2_exp')

In [ ]:
plot_results2(pipe_tf_enet_cv,'ELASTIC NET','score_r2_exp')

In [ ]:
plot_results2(pipe_tf_krr_cv,'KRR','score_r2_exp')

In [ ]:
plot_results2(pipe_tf_svr_cv,'SVR','score_r2_exp')

In [ ]:
plot_results2(pipe_tf_tree_cv,'TREE','score_r2_exp')

In [ ]:
plot_results2(pipe_tf_forest_cv,'FOREST','score_r2_exp')

In [ ]:
plot_results2(pipe_tf_xgb_cv,'GBR','score_r2_exp')

In [ ]:
plot_results2(pipe_tf_ada_cv,'ADA','score_r2_exp')

On expose ici les meilleurs modèles pour chaque type d'estimateur utilisés:

In [ ]:
pipe_tf_ridge_cv.best_params_

In [ ]:
pipe_tf_lasso_cv.best_params_

In [ ]:
pipe_tf_enet_cv.best_params_

In [ ]:
pipe_tf_krr_cv.best_params_

In [ ]:
pipe_tf_svr_cv.best_params_

In [ ]:
pipe_tf_tree_cv.best_params_

In [ ]:
pipe_tf_forest_cv.best_params_

In [ ]:
pipe_tf_xgb_cv.best_params_

In [ ]:
pipe_tf_ada_cv.best_params_

### Evaluation des scores
#### Mean cross-validated score of the best_estimator

In [ ]:
tab_score_gscv = pd.DataFrame({'Mean cv score':[
                                             pipe_tf_ridge_cv.best_score_,
                                             pipe_tf_lasso_cv.best_score_,
                                             pipe_tf_enet_cv.best_score_,
                                             pipe_tf_krr_cv.best_score_,
                                             pipe_tf_svr_cv.best_score_,
                                             pipe_tf_tree_cv.best_score_,
                                             pipe_tf_forest_cv.best_score_,
                                             pipe_tf_xgb_cv.best_score_,
                                             pipe_tf_ada_cv.best_score_]
                         },
            index = ['Ridge','Lasso','ENET','KRR','SVR','TREE','RF','XGB','ADA'])

In [ ]:
# Mean cross-validated score of the best_estimator
tab_score_gscv

In [ ]:
# tab_score_gscv_2 = pd.DataFrame({'Predicator':['Ridge','Lasso','ENET','KRR','SVR','TREE','RF','XGB','ADA'],'Mean cv score':[
#                                              pipe_tf_ridge_cv.best_score_,
#                                              pipe_tf_lasso_cv.best_score_,
#                                              pipe_tf_enet_cv.best_score_,
#                                              pipe_tf_krr_cv.best_score_,
#                                              pipe_tf_svr_cv.best_score_,
#                                              pipe_tf_tree_cv.best_score_,
#                                              pipe_tf_forest_cv.best_score_,
#                                              pipe_tf_xgb_cv.best_score_,
#                                              pipe_tf_ada_cv.best_score_]
#                          })

In [ ]:
# Horizontal barplot of tab_score
tab_score_gscv.plot.barh()
plt.grid()

La plupart des modèles ne dépassent pas les scores de R2 > 0.6.  
Les approches ensemblistes semblent les plus efficaces en moyenne pour prédire les folds de y_train.  
Regardons les variables qui influencent le plus sur ces décisions.  

## Analyse de l'importance ddes variables pour la prédiction (Feature imortance)  
La méthode que l'on va employer ici ne foncitonne que sur les modèles non linéaires de type ensemblistes


In [ ]:
def features_eval(esimator,model_estimator_name,iplot):
    pd_feature_imp = pd.DataFrame([esimator.best_estimator_[model_estimator_name].feature_importances_],columns=trf3.get_feature_names_out(),index=['feature_importances']).T
    if iplot == 0:
        print("DataFrame for feature importances.")
    elif iplot == 1:
        pd_feature_imp[pd_feature_imp != 0].dropna().sort_values(by=['feature_importances'],ascending=False).plot.bar(figsize=([16,10]))
        plt.title('Feature importances evaluation for ' + model_estimator_name)
        plt.grid()
        plt.show()
    return pd_feature_imp

In [ ]:
estimator = [ pipe_tf_ridge_cv,
                 pipe_tf_lasso_cv,
                 pipe_tf_enet_cv,
                 pipe_tf_krr_cv,
                 pipe_tf_svr_cv,
                 pipe_tf_tree_cv,
                 pipe_tf_forest_cv,
                 pipe_tf_xgb_cv,
                 pipe_tf_ada_cv]
estimator_name = ['Ridge','Lasso','ENET','KRR','SVR','TREE','RF','GBR','ADA']
model_name = ['model_ridge','model_lasso','model_enet','model_krr','model_svr','model_tree','model_forest','model_xgb','model_ada']

In [ ]:
for i in range(5,len(estimator)):
    features_eval(estimator[i],model_name[i],1)

On peut voir ici les variables qui ont eu le plus d'importance pour les prédictions, pour chaque modèle.

In [ ]:
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_squared_log_error

def eval_mse(estimator_, estimator_names, X_sample, y_true ):
    #df_mse = pd.DataFrame()
    mse_list = []
    for i in range(len(estimator_names)):
        y_pred = estimator_[i].predict(X_sample)
        mse = mean_squared_error(y_true, y_pred)
        mse_list.append(mse)
        
    df_mse = pd.DataFrame({"Predicateur":estimator_names,"mse":mse_list})
    return df_mse

def eval_msle(estimator_, estimator_names, X_sample, y_true ):
    #df_mse = pd.DataFrame()
    mse_list = []
    for i in range(len(estimator_names)):
        y_pred = estimator_[i].predict(X_sample)
        y_lin = 1 + np.abs( y_true.min() ) +  np.abs( y_pred.min() )
        #print("y_lin = " + str(y_lin))
        y_true_lin = y_true+y_lin.values
        y_pred_lin = y_pred+y_lin.values
        mse = mean_squared_log_error(y_true_lin, y_pred_lin)
        mse_list.append(mse)
        
    df_mse = pd.DataFrame({"Predicateur":estimator_names,"msle":mse_list})
    return df_mse

In [ ]:
mse = eval_mse(estimator, estimator_name, X_train, y_train )
msle = eval_msle(estimator, estimator_name, X_train, y_train )

# Sauvegarde des résultats  
On sauvegarde les résultats pour les comparer aux mêmes modèles entraînés sur d'autres variables.

In [ ]:
# saving the dataframe
tab_score_gscv.to_csv('D:\\Utilisateurs\\Damien\\Documents\\Test_code\\test_python\\OCR_projets\\IML\\P3_\\tab_score_gscv_04.csv', sep='\t', index=True)

In [ ]:
joblib.dump(mse, 'mse_04.pkl')
joblib.dump(msle, 'msle_04.pkl')

In [ ]:
# from sklearn.feature_selection import SelectKBest

# df = SelectKBest(f_regression, k = 5).fit_transform(X_train, y_train)
# print(df.head())

In [ ]:
#import joblib

#save your model or results

# joblib.dump(pipe_tf_ridge_cv, 'pipe_tf_ridge_cv_01.pkl')
# joblib.dump(pipe_tf_lasso_cv, 'pipe_tf_lasso_cv_01.pkl')
# joblib.dump(pipe_tf_enet_cv, 'pipe_tf_enet_cv_01.pkl')
# joblib.dump(pipe_tf_krr_cv, 'pipe_tf_krr_cv_01.pkl')
# joblib.dump(pipe_tf_svr_cv, 'pipe_tf_svr_cv_01.pkl')
# joblib.dump(pipe_tf_tree_cv, 'pipe_tf_tree_cv_01.pkl')
# joblib.dump(pipe_tf_forest_cv, 'pipe_tf_forest_cv_01.pkl')
# joblib.dump(pipe_tf_xgb_cv, 'pipe_tf_xgb_cv_01.pkl')
# joblib.dump(pipe_tf_ada_cv, 'pipe_tf_ada_cv_01.pkl')


joblib.dump(estimator, 'estimator_04.pkl')

On va récupérer le tableau des features sélection.

In [ ]:
def best_feature(estimators,model_name):
    pd_best_feature = pd.DataFrame({'Predicateur':model_name,'feature_name':model_name,'feature_importances':model_name})
    for i in range(len(estimators)):
        pd_feature_imp = features_eval(estimators[i],model_name[i],0)
        pd_best_feature_i = pd_feature_imp[pd_feature_imp['feature_importances'] == pd_feature_imp['feature_importances'].max()]
        pd_best_feature['feature_name'].loc[i] = pd_best_feature_i.index[0]
        pd_best_feature['feature_importances'].loc[i] = pd_best_feature_i.values[0][0]
        
    return pd_best_feature

In [ ]:
pd_best_feature = best_feature(estimator[5:],model_name[5:])
pd_best_feature

In [ ]:
joblib.dump(pd_best_feature, 'pd_best_feature_04.pkl')